# Demo 2 — `mmap`, strides, and who pays

**What you will see:** a transpose that moves zero bytes, why safetensors refuses to save it, and why loading a 1 GB file can cost almost no memory.

Three acts.

In [ ]:
!pip -q install safetensors 2>/dev/null || true

In [ ]:
import torch, os, time, json
from pathlib import Path
from safetensors.torch import save_file, load_file
from safetensors import safe_open
WORK = Path("artifact_demo"); WORK.mkdir(exist_ok=True)

def rss_mb():
    """CURRENT resident set size. Note: resource.ru_maxrss is *peak* and never
    falls, so it cannot show a load costing nothing."""
    try:
        for line in open("/proc/self/status"):
            if line.startswith("VmRSS:"): return int(line.split()[1]) / 1024
    except FileNotFoundError:
        import resource; return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
    return float("nan")

---
## Act 1 — Strides: how a flat buffer pretends to be 2-D

Hardware memory is **one line of addresses**. There is no grid.

In [ ]:
x = torch.arange(6, dtype=torch.float32).reshape(2, 3)
print(x)
print("shape  ", tuple(x.shape))
print("stride ", x.stride())
print("flat   ", x.flatten().tolist())

**Strides `(3, 1)`** — jump 3 elements to change row, 1 to change column. Three columns, so a row is 3 apart.

### The zero-copy transpose

In [ ]:
xt = x.T
print("x.T           ", xt.tolist())
print("x.T.stride()  ", xt.stride())
print("same buffer?  ", x.data_ptr() == xt.data_ptr())
print("contiguous?   ", xt.is_contiguous())

**Same `data_ptr`. Zero bytes moved.** Only shape and strides were swapped.

`x.T` is a **non-contiguous view**: identical memory, different reading rules. Shape alone does not tell you layout.

---
## Act 2 — What making it contiguous actually costs

In [ ]:
big = torch.randn(8192, 8192)                     # 256 MB fp32
print(f"{big.numel()*4/1e6:.0f} MB")

t=time.perf_counter(); v = big.T; t_view = time.perf_counter()-t
t=time.perf_counter(); c = big.T.contiguous(); t_copy = time.perf_counter()-t

print(f"transpose (view)       {t_view*1e6:9.1f} us   same buffer: {v.data_ptr()==big.data_ptr()}")
print(f"transpose + contiguous {t_copy*1e3:9.1f} ms   same buffer: {c.data_ptr()==big.data_ptr()}")
print(f"ratio                  {t_copy/max(t_view,1e-9):9.0f}x")

### Now watch safetensors refuse the view

In [ ]:
for name, tensor in [("contiguous", big), ("transposed view", big.T)]:
    try:
        save_file({"w": tensor}, str(WORK/"probe.safetensors"))
        print(f"{name:<16} -> saved")
    except Exception as e:
        print(f"{name:<16} -> REFUSED: {e}")

**That refusal is the format speaking, not a policy.**

A safetensors header stores `dtype`, `shape` and `data_offsets` — **and no strides**. The format *guarantees* the span is tightly packed row-major, so strides are derivable. A transposed view has no such span to point at.

**Who pays:** the writer. `.contiguous()` runs once on a big machine so that millions of readers only have to `mmap`. **Write once, read millions of times.**

### Read the header yourself — it is plain JSON

In [ ]:
save_file({"a": torch.zeros(2,3), "b": torch.ones(4, dtype=torch.float16)}, str(WORK/"two.safetensors"))
raw = open(WORK/"two.safetensors","rb").read()
n = int.from_bytes(raw[:8], "little")
print("header length:", n, "bytes")
print(json.dumps(json.loads(raw[8:8+n]), indent=2))

---
## Act 3 — `mmap` is not a read

Write a file much larger than anything we intend to use from it.

In [ ]:
save_file({f"layer_{i}": torch.randn(2048, 2048) for i in range(8)},   # 8 x 16 MB
          str(WORK/"big.safetensors"))
size = os.path.getsize(WORK/"big.safetensors")
print(f"file on disk: {size/1e6:.0f} MB")

In [ ]:
before = rss_mb()
with safe_open(WORK/"big.safetensors", framework="pt") as f:
    keys = f.keys()                       # metadata only
    after_keys = rss_mb()
    one = f.get_tensor("layer_0")         # one tensor only
    after_one = rss_mb()

print(f"keys: {list(keys)}")
print(f"RSS before           {before:8.1f} MB")
print(f"RSS after keys       {after_keys:8.1f} MB   (+{after_keys-before:.1f})")
print(f"RSS after 1 tensor  {after_one:8.1f} MB   (+{after_one-after_keys:.1f})")

**Cost scaled with what you read, not with file size.** Now load *everything* and compare.

In [ ]:
before = rss_mb()
allt = load_file(WORK/"big.safetensors")      # every tensor
after = rss_mb()
print(f"load_file (all 8)  RSS +{after-before:6.1f} MB   for a {size/1e6:.0f} MB file")
del allt

### Where do the bytes come from? (Linux only)

`rchar` in `/proc/self/io` counts bytes moved by **`read()` syscalls**.

`mmap` does not call `read()` — it maps pages and lets the fault handler fetch them. So the counter should show the explicit read and stay near zero for the mapped load.

In [ ]:
p = Path("/proc/self/io")
if not p.exists():
    print("/proc/self/io not available on this platform - skip")
else:
    def rchar(): return int(dict(l.split(": ") for l in p.read_text().splitlines())["rchar"])

    a = rchar(); raw = open(WORK/"big.safetensors","rb").read(); b = rchar()
    print(f"explicit read()  rchar +{(b-a)/1e6:7.1f} MB")
    del raw

    a = rchar(); _ = load_file(WORK/"big.safetensors"); b = rchar()
    print(f"safetensors load rchar +{(b-a)/1e6:7.1f} MB")

---
## Takeaway

| Claim | Evidence |
| --- | --- |
| Transpose can move zero bytes | `data_ptr` unchanged; view is ~1000x faster than the copy |
| Formats store shape, not strides | the header JSON has `dtype`, `shape`, `data_offsets` and nothing else |
| The writer pays for contiguity | safetensors refuses the view; `.contiguous()` is a real copy |
| `mmap` cost scales with use | reading one tensor from a 128 MB file barely moved RSS |